# CAP182 SS2 Common Issues & Key Concepts

This notebook addresses four recurring issues from the SS2 assessment.  
We use the **Bank Marketing** dataset from the UCI Machine Learning Repository throughout.

**Contents**
1. Data Leakage — the `duration` trap
2. What `class_weight` actually does
3. Confidence Intervals for a single model
4. Statistical comparison between models

In [0]:
# Install dependencies
!pip install ucimlrepo xgboost -q 

In [0]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    RepeatedStratifiedKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print('All imports successful ✓')

## 1. Load the Bank Marketing dataset

In [0]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=222)
df = dataset.data.original.copy()

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA) on the Bank Marketing dataset

In [0]:
# 1. Basic structure
print('--- Shape ---')
print(df.shape)
print()
print('--- Column names & dtypes ---')
print(df.dtypes)
print()
print('--- First 5 rows ---')
display(df.head())

# 2. Missing values (NaN) per column
print('--- Missing values (NaN) per column ---')
print(df.isna().sum())
print()

# 3. Summary statistics — numeric features
print('--- Numeric summary statistics ---')
display(df.describe())

# 4. Summary statistics — categorical features
print('--- Categorical summary statistics ---')
display(df.describe(include=['object', 'category']))

# 5. Target distribution
y_col = 'y'  # subscription to a term deposit (yes / no)
print(f'--- Target ({y_col}) distribution ---')
print(df[y_col].value_counts())
print()
print(df[y_col].value_counts(normalize=True).round(4) * 100, '%')
print()

# 6. Categorical feature value counts (quick scan)
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    print(f'--- {col} ({df[col].nunique()} unique) ---')
    print(df[col].value_counts())
    print()

# 7. Numeric feature distributions by target
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('--- Numeric features grouped by target ---')
display(df.groupby(y_col)[num_cols].agg(['mean', 'median', 'std']))

# 8. Correlation matrix among numeric features (with legend)
print('--- Correlation matrix (numeric features) ---')
num_df = df.select_dtypes(include=[np.number])
corr = num_df.corr()

styled_corr = (
    corr.style
    .background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1)
    .format(precision=2)
    .set_caption(
        'Correlation matrix — colour scale: 🔴 +1.00 (strong positive) ↔ '
        '⚪ 0.00 (no correlation) ↔ 🔵 −1.00 (strong negative)'
    )
)
display(styled_corr)

# 9. Unknown / placeholder values in categorical columns
print('--- "unknown" value counts in categorical columns ---')
for col in cat_cols:
    unk = (df[col] == 'unknown').sum()
    if unk > 0:
        print(f'{col}: {unk} ({unk / len(df) * 100:.1f}%)')


In [0]:
# 10. Duration vs target — the data-leakage smoking gun
# `duration` is still in df at this point (it gets dropped in preprocessing).
# A long call almost always means the client said "yes", so this column
# leaks the answer into the features.
print('Average call duration by outcome:\n')
print(df.groupby('y')['duration'].agg(['mean', 'median', 'std']).round(1))

## 3. Data Preprocessing

In [0]:
# 3a. Drop the `duration` column — DATA LEAKAGE
# `duration` = length of the last phone call (in seconds).  A call only has a
# duration *after* it ends, and a long call strongly implies the client said
# "yes".  Including it inflates every metric but is impossible to know at
# prediction time, so we remove it.  (This is explored in Section 4 below.)
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
    print('Dropped `duration` (data leakage) ✓')

# 3b. Separate features (X) and target (y)
y_col = 'y'
X = df.drop(columns=[y_col])
y = df[y_col]                     # values: 'yes' / 'no'  (sklearn accepts strings natively)

print(f'\nFeatures shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'Target classes : {y.unique().tolist()}')
print(f'Class counts   :\n{y.value_counts()}')

# 3c. Identify column types
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'\nNumeric features     ({len(numeric_features)}): {numeric_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

# 3d. Build a preprocessing pipeline with ColumnTransformer
#   - Numeric:    median imputation  →  StandardScaler
#   - Categorical: impute 'unknown'  →  OneHotEncoder (handle 'unknown' as its own category)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
)

print('\nPreprocessor built ✓')

# 3e. Stratified train / test split (80 / 20)
# The target is imbalanced (~11 % 'yes'), so stratification preserves the ratio.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'\nTrain set: X_train {X_train.shape}, y_train {y_train.shape}')
print(f'Test  set: X_test  {X_test.shape},  y_test  {y_test.shape}')
print(f'\nTrain class proportions:\n{y_train.value_counts(normalize=True).round(4)}')
print(f'\nTest  class proportions:\n{y_test.value_counts(normalize=True).round(4)}')

# 3f. Fit the preprocessor on the TRAINING data only (avoids leakage)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Retrieve the expanded feature names from the fitted OneHotEncoder
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
processed_feature_names = numeric_features + cat_feature_names

print(f'\nProcessed feature count: {len(processed_feature_names)}')
print(f'Processed train shape : {X_train_processed.shape}')
print(f'Processed test shape  : {X_test_processed.shape}')
print('\nPreprocessing complete ✓')